# Lekcia 08 - Vzorec návrhu viacagentného systému


## Nastavenie


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Prečo viacagentové systémy?

Úlohy zo skutočného sveta, ako napríklad plánovanie výletu, si vyžadujú rôzne druhy odborných znalostí — logistiku, miestne znalosti, rozpočtovanie a ďalšie. Jeden agent, ktorý sa snaží riešiť všetko, sa rýchlo stáva neprehľadným.

Viacagentové systémy to riešia cez **špecializáciu**: každý agent sa sústreďuje na jednu oblasť odbornej expertízy a dosahuje tak kvalitnejšie výsledky než všeobecný agent. Zároveň zlepšujú **škálovateľnosť** — môžete pridať nových agentov (napríklad špecialistu na lety, kritika reštaurácií) bez nutnosti prepísať existujúci pracovný tok. Agenti spolupracujú prostredníctvom štruktúrovaného procesu, pričom odovzdávajú kontext jeden druhému.


## Vytváranie špecializovaných agentov


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Vytváranie sekvenčného pracovného postupu

`WorkflowBuilder` vám umožňuje prepojiť agentov do usmerneného grafu. Tu vytvárame jednoduchý dvojstupňový pipeline: **TravelPlanner** navrhne itinerár a potom ho **TravelConcierge** skontroluje a vylepší.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pridanie ďalších agentov do pracovného postupu

Jednou z najväčších výhod vzoru viacerých agentov je jeho jednoduchá rozšíriteľnosť. Nižšie pridávame agenta **BudgetReviewer**, ktorý kontroluje plán v porovnaní s rozpočtom cestovateľa, označuje položky, ktoré by mohli prekročiť rozpočet, a navrhuje alternatívy šetriace peniaze. Pracovný postup teraz spúšťa troch agentov v poradí:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Zhrnutie

V tejto lekcii ste sa naučili, ako:

1. **Vytvárať špecializovaných agentov** — každý s jasne zameranou úlohou (plánovanie, concierge, kontrola rozpočtu).
2. **Prepojiť agentov do sekvenčného pracovného postupu** pomocou `WorkflowBuilder` a `add_edge`.
3. **Streamovať výstup** z viacagentového reťazca pri sledovaní, ktorý agent práve komunikuje.
4. **Rozšíriť pracovný postup** pridaním nových agentov do reťazca bez úprav existujúcich.

Viacagentový dizajnový vzor udržiava každého agenta jednoduchého a zároveň umožňuje dosiahnuť bohatšie a dôkladnejšie prehodnotené výsledky, než by dokázal jednotlivec sám.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
